In [9]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load dataset (look in current dir or project root)
_csv = Path("budgetwise_finance_dataset_cleaned.csv")
if not _csv.exists():
    _csv = Path.cwd() / "budgetwise_finance_dataset_cleaned.csv"
dataset = pd.read_csv(_csv)
dataset.head()

,transaction_id,user_id,date,transaction_type,category,amount,payment_mode,location,notes
0,T4999,U018,2023-04-25,Expense,Education,3888.0,Card,Ahmedabad,Movie tickets
1,T12828,U133,2022-08-05,Expense,Rent,649.0,NaN,Hyderabad,asdfgh
2,T7403,U091,2023-12-31,Income,Freelance,13239.0,Cash,BAN,Books
3,T7495,U088,2022-10-28,Expense,Entertainment,2287.0,Card,Hyderabad,NaN
4,T12465,U042,2024-04-11,Expense,Food,4168.0,NaN,NaN,test


In [10]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /home/mukama/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [11]:
# Cleaning the texts
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

corpus = []
number_of_rows = dataset.shape[0]
# Use first column for text (works whether it's named "Text", "text", or "Transaction text")
text_col = dataset.columns[0]
for i in range(0, number_of_rows): 
    text = re.sub('[^a-zA-Z0-9]', ' ', str(dataset[text_col].iloc[i])) # use regex to remove all non-alphabetical symbols
    text = text.lower()
    text = text.split()
    ps = PorterStemmer()
    text = [ps.stem(word) for word in text if not word in set(stopwords.words('english'))]
    text = [ps.stem(word) for word in text if not word in set(stopwords.words('danish'))]
    text = ' '.join(text)
    corpus.append(text)

# MORE OPTIONS: 
# 1) remove parts of the string which may contain sensitive information (e.g. contract number) or
#    is described in other features (e.g. date, transaction amount)

In [12]:
# Creating the Bag of Words model
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features = 200)
X = cv.fit_transform(corpus).toarray()
y = dataset.iloc[:, 1]

# Splitting the dataset into the Training set and Test set
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 0)

In [13]:
# Fitting Naive Bayes to the Training set
from sklearn.naive_bayes import GaussianNB
classifier = GaussianNB()
classifier.fit(X_train, y_train)

# Predicting the Test set results
y_pred = classifier.predict(X_test)

# Making the Confusion Matrix
from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(y_test, y_pred)
acc = accuracy_score(y_test, y_pred)
acc

0.015384615384615385

In [14]:
## Logistic regression classification
from sklearn.linear_model import LogisticRegression
classifier = LogisticRegression()
classifier.fit(X_train, y_train)

# Predicting the Test set results
y_pred = classifier.predict(X_test)

# Making the Confusion Matrix
from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(y_test, y_pred)
acc = accuracy_score(y_test, y_pred)
acc

0.019487179487179488

In [15]:
## Random forest tree classification
# Fitting Random Forest Classification to the Training set
from sklearn.ensemble import RandomForestClassifier
classifier = RandomForestClassifier(n_estimators = 10, criterion = 'entropy', random_state = 0)
classifier.fit(X_train, y_train)

# Predicting the Test set results
y_pred = classifier.predict(X_test)

# Making the Confusion Matrix
from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(y_test, y_pred)
acc = accuracy_score(y_test, y_pred)
acc

0.01982905982905983

In [8]:
# Performance comparison: test all models on the test set
import matplotlib.pyplot as plt
import numpy as np
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

models = {
    "Naive Bayes": GaussianNB(),
    "Logistic Regression": LogisticRegression(),
    "Random Forest": RandomForestClassifier(n_estimators=10, criterion="entropy", random_state=0),
}

# Category names for plot labels (optional; uses ids if not all present in data)
category_names = [
    "Automobile", "Housing", "Groceries", "Recreation", "Health",
    "Hobby", "Clothes", "Cash", "Financial", "Other"
]

def plot_confusion_matrix(cm, name, labels=None):
    """Draw confusion matrix as a heatmap."""
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    ax.figure.colorbar(im, ax=ax)
    n = cm.shape[0]
    # Use names only if we have exactly n labels; otherwise use class indices
    if labels is not None and len(labels) >= n:
        tick_labels = labels[:n]
    else:
        tick_labels = [str(i) for i in range(n)]
    ax.set(xticks=np.arange(n), yticks=np.arange(n),
           xticklabels=tick_labels, yticklabels=tick_labels,
           title=f"Confusion matrix — {name}",
           ylabel="True label", xlabel="Predicted label")
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
    for i in range(n):
        for j in range(n):
            ax.text(j, i, int(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black", fontweight="bold")
    fig.tight_layout()
    plt.show()

print("=" * 60)
print("MODEL PERFORMANCE ON TEST SET")
print("=" * 60)

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    print(f"\n--- {name} ---")
    print(f"Accuracy: {acc:.4f}")
    print("\nClassification report (precision, recall, F1 per category):")
    print(classification_report(y_test, y_pred, zero_division=0))
    print("Confusion matrix (rows=true, cols=predicted):")
    print(cm)
    # Draw confusion matrix
    plot_confusion_matrix(cm, name, category_names)

MODEL PERFORMANCE ON TEST SET

--- Naive Bayes ---
Accuracy: 0.0179

Classification report (precision, recall, F1 per category):
              precision    recall  f1-score   support

        U001       0.67      0.10      0.17        20
        U002       0.00      0.00      0.00        21
        U003       0.00      0.00      0.00        24
        U004       0.00      0.00      0.00        27
        U005       0.00      0.00      0.00        18
        U006       0.50      0.06      0.10        18
        U007       0.00      0.00      0.00        28
        U008       0.00      0.00      0.00        15
        U009       0.00      0.00      0.00        18
        U010       0.00      0.00      0.00        22
        U011       0.00      0.00      0.00        19
        U012       1.00      0.05      0.10        19
        U013       0.00      0.00      0.00        24
        U014       0.00      0.00      0.00        26
        U015       0.00      0.00      0.00        16
      

KeyboardInterrupt: 

In [9]:
categories = {
    0:'Automobile and Transport',
    1:'Housing and Real-Estate',
    2:'Groceries',
    3:'Recreation and Leisure',
    4:'Health and Well Being',
    5:'Hobby and Knowledge',
    6:'Clothes and Equipment',
    7:'Cash and Credit',
    8:'Financial Services',
    9:'Other'
}

def get_category_by_id(id):
    return categories[id];

In [21]:
inputs = ['lidl', 'netto ))', 'netflix', 'kfc', 'rent', 'spotify']
predictions = classifier.predict(cv.transform(inputs))
{ inputs[id]: get_category_by_id(predictions[id]) for id in range(predictions.size) }

{'lidl': 'Groceries',
 'netto ))': 'Groceries',
 'netflix': 'Recreation and Leisure',
 'kfc': 'Clothes and Equipment',
 'rent': 'Housing and Real-Estate',
 'spotify': 'Clothes and Equipment'}

In [30]:
# Quick date entry: categorize a free-text statement (e.g. "i spent 4k on netflix sub")
# Use the same preprocessing as training so the model sees consistent text.
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

def preprocess_for_model(raw_text):
    """Same cleaning as training: keep only letters/numbers, lower, stem, remove stopwords (en + da)."""
    text = re.sub(r'[^a-zA-Z0-9]', ' ', str(raw_text))
    text = text.lower().split()
    ps = PorterStemmer()
    en_stop = set(stopwords.words('english'))
    da_stop = set(stopwords.words('danish'))
    text = [ps.stem(w) for w in text if w not in en_stop and w not in da_stop]
    return ' '.join(text)

def predict_category(statement, clf=classifier, vectorizer=cv, category_map=categories):
    """Predict category for one statement (e.g. 'i spent 4k on netflix sub' -> Recreation and Leisure)."""
    preprocessed = preprocess_for_model(statement)
    X = vectorizer.transform([preprocessed])
    pred_id = clf.predict(X)[0]
    return category_map.get(int(pred_id), 'Other')

# Demo: your quick-entry example (entertainment = Recreation and Leisure, id 3)
demo = "i spent 4k on spotify "
print(f"Statement: \"{demo}\"")
print(f"Predicted category: {predict_category(demo)}")

Statement: "i spent 4k on spotify "
Predicted category: Recreation and Leisure
